In [1]:
#import packages and classes
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import seaborn as sns
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
import seaborn as sns
from sklearn import svm
from sklearn.ensemble import RandomForestClassifier#importing ML classes
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_curve
from sklearn.metrics import roc_auc_score
from sklearn import metrics
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

In [2]:
#loading and displaying dataset values
dataset = pd.read_csv(""C:\Users\MANAC Infotech\traffic_data.csv".csv")
dataset

SyntaxError: invalid syntax (Temp/ipykernel_15200/1134420061.py, line 2)

In [ ]:
#plotting graph of traffic flow in different dates
#dataset['time'] = pd.to_datetime(dataset['time'], infer_datetime_format=True)
plt.figure(figsize=(10,4), dpi=100)
plt.plot(dataset.date_time[0:30], dataset.congestion[0:30], color='tab:red')
plt.gca().set(title="Datewise Traffic Congestion", xlabel='date_time', ylabel="Traffic Congestion")
plt.xticks(rotation=90)
plt.show()

In [ ]:
#graphs of different weather condition found in dataset
weather = dataset['direction'].ravel() #extracting weather data
labels, count = np.unique(weather, return_counts=True)
plt.pie(count, labels = labels, autopct='%.0f%%')
plt.title("Distribution of Directions Graph")
plt.show() 

In [ ]:
dataset.groupby(['direction'])['congestion'].sum().plot.barh(figsize=(6,3))
plt.xlabel('Traffic Congestion')
plt.ylabel("Different Directions")
plt.xticks(rotation=90)
plt.show()

In [ ]:
#finding and displaying any missing or null values
dataset.isnull().sum()

In [ ]:
#now convert date column as numeric features by separting them into year, month, day, hour, second and minutes
dataset['date_time'] = pd.to_datetime(dataset['date_time'])
dataset['year'] = dataset['date_time'].dt.year
dataset['month'] = dataset['date_time'].dt.month
dataset['day'] = dataset['date_time'].dt.day
dataset['hour'] = dataset['date_time'].dt.hour
dataset['minute'] = dataset['date_time'].dt.minute
dataset['second'] = dataset['date_time'].dt.second
dataset

In [ ]:
#channels which recive highest comments
data = dataset.groupby(['direction', 'day'])['congestion'].sum().sort_values(ascending=False).reset_index()
sns.catplot(x="direction", y="congestion", hue='day', data=data, kind='point')
plt.title("Daywise Traffic Congestion")
plt.show()

In [ ]:
data = dataset.groupby(['direction', 'month'])['congestion'].sum().sort_values(ascending=False).reset_index()
sns.catplot(x="direction", y="congestion", hue='month', data=data)
plt.title("Month Wise Traffic Congestion")
plt.show()

In [ ]:
#applying label encoder to convert all non-numeric data to numeric values
labels, count = np.unique(dataset['direction'], return_counts=True)
encoder1 = LabelEncoder()
encoder2 = LabelEncoder()
encoder3 = LabelEncoder()
dataset['weather_main'] = pd.Series(encoder1.fit_transform(dataset['weather_main'].astype(str)))#encode all str columns to numeric 
dataset['weather_description'] = pd.Series(encoder2.fit_transform(dataset['weather_description'].astype(str)))#encode all str columns to numeric
dataset['direction'] = pd.Series(encoder3.fit_transform(dataset['direction'].astype(str)))#encode all str columns to numeric 
dataset.drop(['date_time'], axis = 1,inplace=True)
dataset


In [ ]:
#dataset preprocessing and normalization
Y = dataset['direction'].ravel()
dataset.drop(['direction'], axis = 1,inplace=True)
X = dataset.values
sc1 = MinMaxScaler(feature_range = (0, 1))
X = sc1.fit_transform(X)#normalize train features
#split dataset into train and test
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size = 0.2)
print("Total records found in dataset = "+str(X.shape[0]))
print("Total features found in dataset = "+str(X.shape[1]))
print("80% dataset for training : "+str(X_train.shape[0]))
print("20% dataset for testing  : "+str(X_test.shape[0]))
X_train, X_test1, y_train, y_test1 = train_test_split(X, Y, test_size = 0.1)

In [ ]:
#define global variables to save accuracy and other metrics
accuracy = []
precision = []
recall = []
fscore = []

In [ ]:
#function to calculate all metrics
def calculateMetrics(algorithm, testY, predict):
    p = precision_score(testY, predict,average='macro') * 100
    r = recall_score(testY, predict,average='macro') * 100
    f = f1_score(testY, predict,average='macro') * 100
    a = accuracy_score(testY,predict)*100
    accuracy.append(a)
    precision.append(p)
    recall.append(r)
    fscore.append(f)
    print(algorithm+" Accuracy  : "+str(a))
    print(algorithm+" Precision : "+str(p))
    print(algorithm+" Recall    : "+str(r))
    print(algorithm+" FSCORE    : "+str(f))
    conf_matrix = confusion_matrix(testY, predict)
    fig, axs = plt.subplots(1,2,figsize=(10, 4))
    ax = sns.heatmap(conf_matrix, xticklabels = labels, yticklabels = labels, annot = True, cmap="viridis" ,fmt ="g", ax=axs[0]);
    ax.set_ylim([0,len(labels)])
    axs[0].set_title(algorithm+" Confusion matrix") 

    random_probs = [0 for i in range(len(testY))]
    p_fpr, p_tpr, _ = roc_curve(testY, random_probs, pos_label=1)
    plt.plot(p_fpr, p_tpr, linestyle='--', color='orange',label="True classes")
    ns_fpr, ns_tpr, _ = roc_curve(testY, predict, pos_label=1)
    axs[1].plot(ns_tpr, ns_fpr, linestyle='--', label='Predicted Classes')
    axs[1].set_title(algorithm+" ROC AUC Curve")
    axs[1].set_xlabel('False Positive Rate')
    axs[1].set_ylabel('True Positive rate')
    plt.show()

In [ ]:
#train machine learning SVM algorithm to predict route with less traffic
svm_cls = svm.SVC()
svm_cls.fit(X_train, y_train)
predict = svm_cls.predict(X_test)
calculateMetrics("SVM", y_test, predict)

In [ ]:
#train machine learning Decision Tree algorithm to predict route with less traffic
dt_cls = DecisionTreeClassifier()
dt_cls.fit(X_train, y_train)
#perform prediction on test data
predict = dt_cls.predict(X_test)
#calculate prediction accuracy and other metrics
calculateMetrics("Decision Tree", y_test, predict)

In [ ]:
#train machine learning Random Forest algorithm to predict route with less traffic
rf_cls = RandomForestClassifier()
rf_cls.fit(X_train, y_train)
#perform prediction on test data
predict = rf_cls.predict(X_test)
#calculate prediction accuracy and other metrics
calculateMetrics("Random Forest", y_test, predict)

In [ ]:
#plot all algorithm performance in tabukar format
df = pd.DataFrame([['SVM','Accuracy',accuracy[0]],['SVM','Precision',precision[0]],['SVM','Recall',recall[0]],['SVM','FSCORE',fscore[0]],
                   ['Decision Tree','Accuracy',accuracy[1]],['Decision Tree','Precision',precision[1]],['Decision Tree','Recall',recall[1]],['Decision Tree','FSCORE',fscore[1]],
                   ['Random Forest','Accuracy',accuracy[2]],['Random Forest','Precision',precision[2]],['Random Forest','Recall',recall[2]],['Random Forest','FSCORE',fscore[2]],
                  ],columns=['Parameters','Algorithms','Value'])
df.pivot("Parameters", "Algorithms", "Value").plot(kind='bar', figsize=(6, 3))
plt.title("All Algorithms Performance Graph")
plt.show()

In [ ]:
#display all algorithm performnace
algorithms = ['SVM', 'Decision Tree', 'Random Forest']
data = []
for i in range(len(accuracy)):
    data.append([algorithms[i], accuracy[i], precision[i], recall[i], fscore[i]])
data = pd.DataFrame(data, columns=['Algorithm Name', 'Accuracy', 'Precision', 'Recall', 'FSCORE'])
data    

In [ ]:
#function to perform route prediction on test data using congestion
#reading test data from test file and then predicting traffic volume
dataset = pd.read_csv("Dataset/testData.csv")
dataset.fillna(0, inplace = True) #remove missing values
temp = dataset.values
dataset['date_time'] = pd.to_datetime(dataset['date_time'])#convert column to date time
dataset['year'] = dataset['date_time'].dt.year #converting date into year, month
dataset['month'] = dataset['date_time'].dt.month
dataset['day'] = dataset['date_time'].dt.day
dataset['hour'] = dataset['date_time'].dt.hour
dataset['minute'] = dataset['date_time'].dt.minute
dataset['second'] = dataset['date_time'].dt.second
dataset.drop(['date_time'], axis = 1,inplace=True)
dataset['weather_main'] = pd.Series(encoder1.transform(dataset['weather_main'].astype(str)))#encode all str columns to numeric 
dataset['weather_description'] = pd.Series(encoder2.transform(dataset['weather_description'].astype(str)))#encode all str columns to numeric
testData = dataset.values
#normalizing test data
testData = sc1.transform(testData)
#perform prediction on test
predict = dt_cls.predict(testData)
for i in range(len(predict)):
    print("Test Data = "+str(temp[i])+" Suggested Route is : "+labels[predict[i]])
    print()

In [ ]:
def predictAccidentSeverity(rf, label_encoder, scaler1, column):
    dataset = pd.read_csv("Dataset/accident_testData.csv")
    column = dataset.columns.ravel()
    index = 0
    for i in range(len(column)):
        if str(dataset.dtypes[column[i]]) == 'object' and column[i] != 'Severity':
            dataset[column[i]] = pd.Series(label_encoder[index].fit_transform(dataset[column[i]].astype(str)))
            index = index + 1
        if str(dataset.dtypes[column[i]]) == 'bool':
            dataset[column[i]] = dataset[column[i]].astype(int)
    dataset.fillna(0, inplace = True)        
    dataset = dataset.apply(lambda x: x.fillna(x.mean()))        
    test = dataset.values
    test = test[:,0:test.shape[1]]
    dataset = scaler1.transform(test)
    predict = rf.predict(dataset)
    for i in range(len(predict)):
        print("Test Data = "+str(test[i])+" Predicted Severity Level ===> "+str(predict[i])+"\n")

In [ ]:
#accident severity detection module
dataset = pd.read_csv("Dataset/US_Accidents_Dec21_updated.csv")
column = dataset.columns.ravel()
label_encoder = []
for i in range(len(column)):
    if str(dataset.dtypes[column[i]]) == 'object':
        le = LabelEncoder()
        dataset[column[i]] = pd.Series(le.fit_transform(dataset[column[i]].astype(str)))
        label_encoder.append(le)
    if str(dataset.dtypes[column[i]]) == 'bool':
        dataset[column[i]] = dataset[column[i]].astype(int)
Y = dataset['Severity'].ravel()
dataset.drop(['Severity'], axis = 1,inplace=True)      
dataset = dataset.apply(lambda x: x.fillna(x.mean()))
scaler1 = MinMaxScaler()
X = dataset.values
X = scaler1.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size = 0.2)
rf = RandomForestClassifier()
rf.fit(X_train, y_train)
predict = rf.predict(X_test)
print("Accident Training Model Completed")

In [ ]:
predictAccidentSeverity(rf, label_encoder, scaler1, column)